# Data Skew: Simulating Uniform vs Skewed Datasets

**What this notebook does**

* Builds a **uniform** DataFrame and shows that rows are spread evenly across partitions.
* Builds a **skewed** DataFrame where almost all rows land in a single partition.
* Reproduces a real-world **join skew** using `transactions` and `customers` parquet data, where a few customers own a disproportionate share of transactions.

**Core concept: data skew**

Spark processes each partition as one task. If one partition holds far more data than the others, its task runs much longer while the remaining cores sit idle. The stage (and the whole job) only finishes when the slowest task finishes, so a single heavy partition dictates total runtime and can cause spills or out-of-memory errors.

**Databricks conversion notes**

* Originally written for local Jupyter (`local[4]`). On Databricks the `spark` session is pre-created, so the `SparkSession.builder` / `sparkContext` code was removed (standard access mode and serverless use Spark Connect, which has no `sparkContext`).
* Local `../../data/data_skew` paths were replaced with the Unity Catalog volume `/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed`.
* Recommended compute: **Sandani Personal Compute** (classic cluster). On serverless, `spark.sql.adaptive.enabled` and `spark.sql.autoBroadcastJoinThreshold` cannot be changed (the cells catch this and print a message), so AQE and broadcast joins stay on and skew effects may be hidden.

## 1. Imports & Configuration

In [0]:
# Note: the Jupyter-only InteractiveShell setting was removed; Databricks renders cell output natively.
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')
import time

In [0]:
# SparkSession is not imported: Databricks provides a ready-to-use `spark` object.
from pyspark.sql.types import IntegerType
import pyspark.sql.functions as F

In [0]:
# Original (local Jupyter):
#   spark = SparkSession.builder.master("local[4]").getOrCreate()
#   sc = spark.sparkContext
#   sc.setLogLevel("ERROR")
# On Databricks the session already exists and sparkContext is not available under Spark Connect.
print(f"Spark version: {spark.version}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

**What:** Turn off Adaptive Query Execution (AQE).

**Core concept:** AQE re-optimizes queries at runtime using shuffle statistics. It can coalesce small partitions and automatically split skewed join partitions (`spark.sql.adaptive.skewJoin.enabled`). Disabling it lets us observe skew in its raw form; later notebooks (`3_solving_data_skew_aqe_broadcast`) turn it back on as a fix.

In [0]:
# spark.conf.set("spark.sql.shuffle.partitions", "3")
try:
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    print("AQE disabled")
except Exception as e:
    # Serverless does not allow changing this config; run on a classic cluster to see raw skew.
    print(f"Could not disable AQE on this compute: {type(e).__name__}")

## 2. Simulating a Uniform Dataset

**What:** `spark.range(1000000)` creates 1M rows with a single `id` column.

**Core concept:** `spark.range` splits the id range evenly across the default parallelism (number of available cores). Each partition therefore gets roughly the same number of rows, which is the ideal case: all tasks do equal work and finish at about the same time.

In [0]:
df_uniform = spark.range(1000000)
display(df_uniform.limit(3))

**What:** Count rows per partition.

**Core concept:** `F.spark_partition_id()` tags each row with the id of the partition it lives in. Grouping by that id is a quick way to measure how evenly data is distributed. Expect near-equal counts here. The number of partitions depends on the cores of the attached compute (the original local run used `local[4]`, giving 4 x 250,000).

In [0]:
(
    df_uniform
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .count()
    .orderBy("partition")
    .show()
)

## 3. Simulating a Skewed Dataset

**What:** Three DataFrames are each forced into a single partition with `repartition(1)` and then combined with `union`:

* `df0` with 1,000,000 rows
* `df1` and `df2` with 10 rows each

**Core concept:** `union` does not shuffle; it simply concatenates the partitions of its inputs. The result has 3 partitions of very unequal size (1,000,000 vs 10 vs 10). Any per-partition work on `df_skew` is dominated by the one heavy partition, which is exactly what data skew looks like in the Spark UI: one long-running task and many short ones.

In [0]:
df0 = spark.range(0, 1000000).repartition(1)
df1 = spark.range(0, 10).repartition(1)
df2 = spark.range(0, 10).repartition(1)
df_skew = df0.union(df1).union(df2)
display(df_skew.limit(3))

**What:** Count rows per partition of `df_skew`.

**Core concept:** Expect partition 0 to hold 1,000,000 rows and partitions 1 and 2 to hold 10 each. Open the Spark UI (cluster > Spark UI > Stages) and compare task durations: one task does almost all the work.

In [0]:
(
    df_skew
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition")
    .count()
    .orderBy("partition")
    .show()
)

## 4. Join Skews

**What:** Load a transactions fact table and a customers dimension table, then join them on `cust_id`.

**Core concept:** In a sort-merge join, Spark shuffles both sides so that all rows with the same join key end up in the same partition. If a handful of keys (here, a few `cust_id`s) have far more rows than the rest, the partitions holding those keys become huge. This is **join skew**, the most common form of skew in practice.

In [0]:
# Local paths replaced with the Unity Catalog volume
base_path = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed"
transactions_file = f"{base_path}/transactions.parquet"
customer_file = f"{base_path}/customers.parquet"

df_transactions = spark.read.parquet(transactions_file)
df_customers = spark.read.parquet(customer_file)

**What:** Inspect the schema and a sample of each table.

**Core concept:** `printSchema()` reads only parquet metadata (cheap), while `display(... .limit(5))` triggers a small job. `cust_id` is the join key shared by both tables.

In [0]:
df_transactions.printSchema()
display(df_transactions.limit(5))

In [0]:
df_customers.printSchema()
display(df_customers.limit(5))

**What:** Count distinct transactions per customer and list the top 5.

**Core concept:** Before fixing skew, confirm it. A key-frequency check like this reveals the "hot" keys. If the top customers have orders of magnitude more transactions than typical customers, a join on `cust_id` will produce a few oversized shuffle partitions.

In this dataset one customer (`C0YDPQWPBJ`) has about 17.5M transactions while the next ones have about 8K, so its shuffle partition is thousands of times larger than the rest.

In [0]:
df_top_customers = (
    df_transactions
    .groupBy("cust_id")
    .agg(F.countDistinct("txn_id").alias("ct"))
    .orderBy(F.desc("ct"))
    .limit(5)
)
display(df_top_customers)

**What:** Disable automatic broadcast joins.

**Core concept:** When one side of a join is small (below `spark.sql.autoBroadcastJoinThreshold`, 10 MB by default), Spark broadcasts it to every executor and avoids a shuffle, which sidesteps skew entirely. Setting the threshold to `-1` forces a **sort-merge join**, so both sides are shuffled by `cust_id` and the skew becomes visible.

In [0]:
try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
    print("Broadcast joins disabled")
except Exception as e:
    print(f"Could not change autoBroadcastJoinThreshold on this compute: {type(e).__name__}")

**What:** Define an inner join of transactions and customers on `cust_id`.

**Core concept:** Transformations are **lazy**: this cell only builds a logical plan; no data is read or shuffled until an action is called.

In [0]:
df_txn_details = (
    df_transactions.join(
        df_customers,
        on="cust_id",
        how="inner"
    )
)

**What:** Trigger the join with `count()` and measure wall-clock time.

**Core concept:** `count()` is an action, so Spark executes the full plan: scan, shuffle both sides, sort-merge join. In the Spark UI, the join stage should show most tasks finishing quickly and a few long tasks processing the hot `cust_id` partitions. This runtime is the baseline that the AQE, broadcast and salting notebooks improve on.

In [0]:
start_time = time.time()
row_count = df_txn_details.count()
print(f"rows: {row_count:,}")
print(f"time taken: {time.time() - start_time}")

In [0]:
# spark.stop()
# Do not stop the session on Databricks: the session is managed by the compute and is shared by the notebook.

## Summary

* Uniform data: equal rows per partition, balanced tasks.
* Skewed data: one partition holds nearly all rows, so one task dominates stage runtime.
* Join skew: hot join keys concentrate rows into a few shuffle partitions during a sort-merge join.

**Next:** `3_solving_data_skew_aqe_broadcast` (AQE skew join and broadcast joins) and `4_salting` (key salting).